# Notebook 06 — How real systems compare

Notebook 05 listed *what* we omitted. This chapter places pq-messenger on a map of real deployed protocols: Signal's Double Ratchet, Apple iMessage PQ3, WireGuard, and MLS. The point isn't to score them — it's to see *which design problem each one is solving* and where pq-messenger sits as a teaching tool.

## The four systems on one page

| Aspect                       | **pq-messenger**            | **Signal (Double Ratchet)** | **iMessage PQ3**          | **WireGuard**                  | **MLS (RFC 9420)**             |
| ---------------------------- | --------------------------- | --------------------------- | ------------------------- | ------------------------------ | ------------------------------ |
| Primary use case             | Teaching, 1:1 demo          | 1:1 messaging               | 1:1 + small group iMsg    | VPN tunnel (point-to-point)    | Large group messaging          |
| Initial key agreement        | Hybrid X3DH (X25519+ML-KEM) | X3DH (X25519, classical)    | Hybrid (ECDH+ML-KEM)      | Noise IK (X25519)              | Tree-based, per-group          |
| Continuous rekey             | Symmetric only              | Double ratchet (DH+sym)     | DR + periodic PQ rekey    | Per-handshake, every ~2 min    | Tree-based ratchet             |
| Post-quantum?                | Yes (KEM half hybrid)       | Not yet (planned)           | Yes (since iOS 17.4)      | No (yet)                       | Pluggable; PQ ciphersuites WIP |
| Authentication               | TOFU                        | Ed25519 + safety numbers    | Apple ID + Contact Verif. | Pre-shared static public keys  | Credentials + tree signatures  |
| Group chats                  | No                          | Sender Keys                 | Pairwise sessions         | N/A                            | Native, scalable to 1000s      |
| Multi-device                 | No                          | Per-device sessions         | iCloud Keychain sync      | One key per peer (manual)      | Per-leaf in the group tree     |
| Metadata privacy             | None                        | Sealed sender               | Apple ID minimization     | Server-less by design          | Out of scope (transport job)   |
| Async (offline) delivery     | File poll                   | Server queue + push         | APNs                      | No (always-on tunnel)          | Server-mediated                |
| Standard                     | This notebook               | Signal spec                 | Apple internal            | RFC informational              | RFC 9420                       |

## Pairwise observations

### pq-messenger vs Signal Double Ratchet

The **shape** is the same: long-term identity, ephemeral handshake, root key, per-direction chain key, AEAD per message. The **gap** is exactly the DH ratchet — see notebook 05 §1. Add ~150 lines and pq-messenger turns into a (post-quantum, pure-Python) Signal.

What pq-messenger has that vanilla Signal does *not* (yet): a post-quantum half in the initial handshake. Signal published a [PQXDH proposal](https://signal.org/docs/specifications/pqxdh/) in 2023 that does the same hybrid step we use; rollout is in progress.

### pq-messenger vs iMessage PQ3

PQ3 is what a *production* hybrid-PQ messenger looks like. Apple's [tech report](https://security.apple.com/blog/imessage-pq3/) (2024) is the closest real-world analogue to what we're teaching here, with three levers we don't pull:

1. **Periodic PQ rekey** — every N messages or M minutes, run a fresh ML-KEM encapsulation to refresh entropy. Closes the "harvest now, decrypt later" attack window even mid-conversation.
2. **Contact Key Verification** — surfaces identity-key changes to the user. The cure for TOFU.
3. **Hybrid commitment in the AEAD** — the AEAD's associated data binds the message to *both* the classical and PQ secrets, so a future break of one half can't retroactively forge messages.

### pq-messenger vs WireGuard

WireGuard is a **tunnel**, not a messenger. The handshake (Noise IK) runs *every ~2 minutes* by default, so post-compromise security is achieved by re-handshaking rather than ratcheting. Cryptographically simpler — no skipped-key cache, no ratchet state — but pays for it with bandwidth and a long-lived TCP/UDP connection assumption. WireGuard chose to *not* be post-quantum; the project's stance is "wait for PQ KEM standards to mature".

### pq-messenger vs MLS

Different problem. MLS (RFC 9420) is built for groups of thousands; the "ratchet" is a binary tree where adding/removing a member updates only $O(\log N)$ keys. WhatsApp and Discord moved from Sender Keys to MLS for exactly this reason. For two parties MLS is overkill; pq-messenger sits at the opposite end of the size spectrum.

## What this map is good for

When you read a real protocol spec — Signal's, Apple's, IETF's — you'll see hundreds of pages on identity provisioning, multi-device, abuse reporting, group membership management, and key transparency. Most of that is **orthogonal to the cryptographic core**. The core is what we built:

- A handshake that survives both classical and quantum adversaries.
- A ratchet that protects past messages even if the present is compromised.
- An AEAD that detects every byte of tampering.

Once those three are working, the rest is engineering — important, hard, but a different kind of hard. Notebook 05 is the inventory of what "the rest" actually contains.

## References

- Signal Foundation. [*The Double Ratchet Algorithm*](https://signal.org/docs/specifications/doubleratchet/). 2016.
- Signal Foundation. [*The X3DH Key Agreement Protocol*](https://signal.org/docs/specifications/x3dh/). 2016.
- Signal Foundation. [*PQXDH — A Post-Quantum Extended Diffie-Hellman Key Agreement*](https://signal.org/docs/specifications/pqxdh/). 2023.
- Apple Security Engineering. [*iMessage with PQ3: The new state of the art in quantum-secure messaging at scale*](https://security.apple.com/blog/imessage-pq3/). 2024.
- Donenfeld, Jason. [*WireGuard: Next Generation Kernel Network Tunnel*](https://www.wireguard.com/papers/wireguard.pdf). NDSS 2017.
- IETF MLS WG. [*The Messaging Layer Security (MLS) Protocol*](https://datatracker.ietf.org/doc/html/rfc9420). RFC 9420. 2023.